In [ ]:
"""
Amortization / Cash Flow Model for Loan Repline Pricing

Replicates the Excel amortization sheet with:
- Standard amortization with prepayments (CPR/SMM) and defaults (CDR/MDR)
- 4-month chargeoff/recovery lag
- PV discounting at a given discount rate
- Goal seek: find CDR that produces a target cumulative gross loss (CGL)

Inputs:
    UPB, WAC, Amortization Term, Severity, Discount Rate, CPR, CDR (or Target CGL)

Key Outputs:
    Repline Price (PV / UPB), Total PV, CGL, full amortization table
"""

import numpy as np
from scipy.optimize import brentq
import pandas as pd

CHARGEOFF_LAG = 4


def build_amort_table(upb, wac, term, severity, discount_rate, cpr, cdr):
    """
    Build the full amortization table and compute summary metrics.

    Parameters
    ----------
    upb : float            Current unpaid principal balance
    wac : float            Weighted average coupon (annual, e.g. 0.1439)
    term : int             Amortization term in months
    severity : float       Loss severity (e.g. 0.9 = 90% loss, 10% recovery)
    discount_rate : float  Annual discount rate for PV (e.g. 0.075)
    cpr : float            Constant prepayment rate (annual, e.g. 0.12)
    cdr : float            Constant default rate (annual, e.g. 0.0096)

    Returns
    -------
    df : pd.DataFrame      Period-by-period amortization table
    summary : dict          Aggregate metrics (Price, PV, CGL, totals)
    """
    smm = 1 - (1 - cpr) ** (1 / 12)
    mdr = 1 - (1 - cdr) ** (1 / 12)
    monthly_rate = wac / 12
    total_periods = term + CHARGEOFF_LAG

    starting_bal = np.zeros(total_periods + 1)
    ending_bal = np.zeros(total_periods + 1)
    sched_payment = np.zeros(total_periods + 1)
    interest = np.zeros(total_periods + 1)
    sched_prin = np.zeros(total_periods + 1)
    actual_prin = np.zeros(total_periods + 1)
    defaults_arr = np.zeros(total_periods + 1)
    chargeoffs = np.zeros(total_periods + 1)
    recoveries = np.zeros(total_periods + 1)
    cashflow = np.zeros(total_periods + 1)
    pv_cf = np.zeros(total_periods + 1)

    starting_bal[0] = upb
    ending_bal[0] = upb

    for n in range(1, total_periods + 1):
        # Defaults: MDR applied to prior period's ending balance
        if n <= term:
            defaults_arr[n] = mdr * ending_bal[n - 1]

        # Chargeoffs and recoveries arrive after the lag
        if n > CHARGEOFF_LAG:
            chargeoffs[n] = -defaults_arr[n - CHARGEOFF_LAG]
            recoveries[n] = (1 - severity) * defaults_arr[n - CHARGEOFF_LAG]

        # Starting balance: prior ending, minus current defaults leaving pool,
        # plus current recoveries re-entering pool
        if n == 1:
            starting_bal[n] = ending_bal[n - 1]
        else:
            starting_bal[n] = ending_bal[n - 1] - defaults_arr[n] + recoveries[n]

        remaining = term - n + 1
        if remaining > 0 and ending_bal[n - 1] > 0.01:
            sched_payment[n] = (
                ending_bal[n - 1]
                * monthly_rate
                / (1 - (1 + monthly_rate) ** (-remaining))
            )
            interest[n] = (1 - mdr) * monthly_rate * ending_bal[n - 1]
            sched_prin[n] = (
                sched_payment[n]
                - interest[n]
                - ending_bal[n - 1] * mdr * monthly_rate
            )
            ending_bal[n] = (ending_bal[n - 1] - sched_prin[n]) * (1 - smm - mdr)
        else:
            ending_bal[n] = 0.0

        actual_prin[n] = (
            ending_bal[n - 1] - ending_bal[n] - defaults_arr[n] + recoveries[n]
        )
        cashflow[n] = actual_prin[n] + interest[n]
        pv_cf[n] = cashflow[n] / (1 + discount_rate) ** (n / 12)

    periods = np.arange(total_periods + 1)
    df = pd.DataFrame(
        {
            "Period": periods,
            "Starting Balance": starting_bal,
            "Ending Balance": ending_bal,
            "Scheduled Payment": sched_payment,
            "Interest": interest,
            "Scheduled Principal": sched_prin,
            "Actual Principal": actual_prin,
            "Defaults": defaults_arr,
            "Chargeoffs": chargeoffs,
            "Recoveries": recoveries,
            "Cash Flow": cashflow,
            "PV": pv_cf,
            "CPR": np.where(periods > 0, cpr, 0.0),
            "SMM": np.where(periods > 0, smm, 0.0),
            "CDR": np.where(periods > 0, cdr, 0.0),
            "MDR": np.where(periods > 0, mdr, 0.0),
        }
    )

    total_pv = np.sum(pv_cf)
    summary = {
        "Total PV": total_pv,
        "Price": total_pv / upb,
        "CGL": np.sum(defaults_arr) / upb,
        "Total Interest": np.sum(interest),
        "Total Scheduled Principal": np.sum(sched_prin),
        "Total Defaults": np.sum(defaults_arr),
        "Total Chargeoffs": np.sum(chargeoffs),
        "Total Recoveries": np.sum(recoveries),
        "Total Cash Flow": np.sum(cashflow),
    }
    return df, summary


def goal_seek_cdr(upb, wac, term, severity, discount_rate, cpr, target_cgl):
    """
    Find the CDR that produces the target cumulative gross loss (CGL).

    Uses Brent's method (bisection + secant) for robust root finding.
    """

    def objective(cdr):
        _, s = build_amort_table(upb, wac, term, severity, discount_rate, cpr, cdr)
        return s["CGL"] - target_cgl

    return brentq(objective, 1e-8, 0.99, xtol=1e-12)


def compute_yield(upb, wac, term, severity, cpr, cdr, target_price):
    """
    Compute the yield (discount rate) that makes PV equal to target_price * UPB.

    This is the IRR from the buyer's perspective: what discount rate equates
    the present value of all future cash flows to the purchase price?
    """

    def objective(dr):
        _, s = build_amort_table(upb, wac, term, severity, dr, cpr, cdr)
        return s["Price"] - target_price

    return brentq(objective, 1e-6, 1.0, xtol=1e-12)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    # ── Inputs ──
    upb = 4_217_833.27
    wac = 0.1439
    term = 180
    severity = 0.9
    discount_rate = 0.0575
    cpr = 0.12
    target_cgl = 0.05

    # ── Goal-seek CDR to hit target CGL ──
    cdr = goal_seek_cdr(upb, wac, term, severity, discount_rate, cpr, target_cgl)

    # ── Build amortization table ──
    df, summary = build_amort_table(upb, wac, term, severity, discount_rate, cpr, cdr)

    # ── Display results ──
    print("=" * 60)
    print("  LOAN REPLINE CASH FLOW MODEL")
    print("=" * 60)

    print("\nINPUTS")
    print("-" * 40)
    print(f"  UPB:              ${upb:>15,.2f}")
    print(f"  WAC:              {wac:>15.4%}")
    print(f"  Term:             {term:>12d} months")
    print(f"  Severity:         {severity:>15.2%}")
    print(f"  Discount Rate:    {discount_rate:>15.4%}")
    print(f"  CPR:              {cpr:>15.4%}")
    print(f"  Target CGL:       {target_cgl:>15.4%}")

    smm = 1 - (1 - cpr) ** (1 / 12)
    mdr = 1 - (1 - cdr) ** (1 / 12)
    print(f"\n  Solved CDR:       {cdr:>15.10%}")
    print(f"  SMM:              {smm:>15.10%}")
    print(f"  MDR:              {mdr:>15.10%}")

    print(f"\n{'=' * 60}")
    print("  OUTPUTS")
    print("=" * 60)
    print(f"  Repline Price:    {summary['Price']:>18.6f}")
    print(f"  Repline PV:       ${summary['Total PV']:>17,.2f}")
    print(f"  CGL:              {summary['CGL']:>18.6%}")
    print(f"  Total Interest:   ${summary['Total Interest']:>17,.2f}")
    print(f"  Total Sched Prin: ${summary['Total Scheduled Principal']:>17,.2f}")
    print(f"  Total Defaults:   ${summary['Total Defaults']:>17,.2f}")
    print(f"  Total Chargeoffs: ${summary['Total Chargeoffs']:>17,.2f}")
    print(f"  Total Recoveries: ${summary['Total Recoveries']:>17,.2f}")
    print(f"  Total Cash Flow:  ${summary['Total Cash Flow']:>17,.2f}")

    # ── Amortization table ──
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 220)
    pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

    print(f"\n{'=' * 60}")
    print("  AMORTIZATION TABLE  (periods 0 - 6)")
    print("=" * 60)
    print(df.head(7).to_string(index=False))

    print(f"\n{'=' * 60}")
    print(f"  AMORTIZATION TABLE  (periods {term - 2} - {term + CHARGEOFF_LAG})")
    print("=" * 60)
    print(df.iloc[term - 2 :].to_string(index=False))

    # ── Optional: compute yield for a target price ──
    target_price = summary["Price"]
    yld = compute_yield(upb, wac, term, severity, cpr, cdr, target_price)
    print(f"\n{'=' * 60}")
    print(f"  Yield at Price {target_price:.6f}: {yld:.6%}")
    print("=" * 60)


  LOAN REPLINE CASH FLOW MODEL

INPUTS
----------------------------------------
  UPB:              $   4,217,833.27
  WAC:                     14.3900%
  Term:                      180 months
  Severity:                  90.00%
  Discount Rate:            7.5000%
  CPR:                     12.0000%
  Target CGL:               5.0000%

  Solved CDR:         0.9655053480%
  SMM:                1.0596241035%
  MDR:                0.0808170397%

  OUTPUTS
  Repline Price:              1.245370
  Repline PV:       $     5,252,761.89
  CGL:                       5.000000%
  Total Interest:   $     3,126,690.52
  Total Sched Prin: $     1,256,183.81
  Total Defaults:   $       210,891.66
  Total Chargeoffs: $      -210,891.66
  Total Recoveries: $        21,089.17
  Total Cash Flow:  $     7,154,721.29

  AMORTIZATION TABLE  (periods 0 - 6)
 Period  Starting Balance  Ending Balance  Scheduled Payment    Interest  Scheduled Principal  Actual Principal   Defaults  Chargeoffs  Recoveries    Cas

In [17]:
from pathlib import Path
import os


def _norm_col(c):
    return "".join(ch.lower() for ch in str(c) if ch.isalnum())


def _pick_col(df, aliases, required=True):
    norm_map = {_norm_col(c): c for c in df.columns}
    for alias in aliases:
        k = _norm_col(alias)
        if k in norm_map:
            return norm_map[k]
    if required:
        raise KeyError(f"Missing required column. Tried aliases: {aliases}")
    return None


def _to_num(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().replace(",", "").replace("$", "")
        if s.endswith("%"):
            s = s[:-1]
        return pd.to_numeric(s, errors="coerce")
    return pd.to_numeric(x, errors="coerce")


def _as_rate(x):
    x = _to_num(x)
    if pd.isna(x):
        return np.nan
    # If user data is in percent points (e.g., 10.25), convert to decimal (0.1025).
    return x / 100.0 if x > 1.0 else x


def _candidate_paths():
    home = Path.home()
    return [
        home / "replines fico lb purpose.xlsx",
        home / "Downloads" / "replines fico lb purpose.xlsx",
        home / "Desktop" / "replines fico lb purpose.xlsx",
        home / "Documents" / "replines fico lb purpose.xlsx",
    ]


def _auto_find_repline_file(start_dir=Path.home()):
    # First try exact likely attachment filename in common folders.
    for p in _candidate_paths():
        if p.exists():
            return p

    # Then try keyword matching.
    keywords = ["repline", "fico", "purpose"]
    for p in start_dir.rglob("*.xlsx"):
        name = p.name.lower()
        if all(k in name for k in keywords):
            return p

    # Fallback to any workbook with 'repline' in name.
    for p in start_dir.rglob("*.xlsx"):
        if "repline" in p.name.lower():
            return p

    return None


def _read_excel_detect_header(file_path, sheet_name):
    # Some workbooks have metadata rows above headers; try several header offsets.
    for header_row in range(0, 12):
        try:
            trial = pd.read_excel(file_path, sheet_name=sheet_name, header=header_row)
        except Exception:
            continue

        norm_cols = {_norm_col(c) for c in trial.columns}
        has_upb = any(k in norm_cols for k in [
            _norm_col("UPB"),
            _norm_col("Current UPB"),
            _norm_col("Total Current Principal"),
            _norm_col("Total Orig Principal"),
        ])
        has_wac = any(k in norm_cols for k in [
            _norm_col("WAC"),
            _norm_col("Weighted Avg Interest"),
            _norm_col("WA WAC"),
        ])
        has_term = any(k in norm_cols for k in [
            _norm_col("Term"),
            _norm_col("Term.1"),
            _norm_col("Weighted Avg Term"),
            _norm_col("Remaining Term"),
        ])

        if has_upb and has_wac and has_term:
            return trial

    # Fallback default behavior.
    return pd.read_excel(file_path, sheet_name=sheet_name)


def run_replines_from_excel(
    file_path=None,
    sheet_name=0,
    output_path=None,
    default_severity=0.90,
    default_discount_rate=0.075,
    default_cpr=0.12,
):
    """
    Batch-run replines from Excel.

    Expected input columns (flexible naming):
    - Required: UPB, WAC, Term
    - Optional direct CDR: CDR
    - Optional goal seek: Target CGL
    - Optional overrides: Severity, Discount Rate, CPR
    - Optional id columns: Repline ID / Loan ID / Purpose / FICO
    """
    if file_path is None:
        env_path = os.environ.get("REPLINE_FILE")
        if env_path:
            file_path = Path(env_path)
        else:
            found = _auto_find_repline_file()
            if found is None:
                raise FileNotFoundError(
                    "Could not auto-find the repline file. "
                    "Set REPLINE_FILE env var or pass file_path explicitly."
                )
            file_path = found

    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Input file not found: {file_path}")

    df_in = _read_excel_detect_header(file_path, sheet_name)

    upb_col = _pick_col(
        df_in,
        [
            "Total Current Principal",
            "Current Principal",
            "Total Orig Principal",
            "Current UPB",
            "Original Balance",
            "Balance",
            "UPB",
        ],
    )
    wac_col = _pick_col(
        df_in,
        [
            "WA WAC",
            "Weighted Avg Interest",
            "WA Interest",
            "WAC",
            "Coupon",
            "Rate",
            "Interest Rate",
        ],
    )
    term_col = _pick_col(
        df_in,
        ["Term.1", "Weighted Avg Term", "Remaining Term", "Amort Term", "Amortization Term", "Term"],
    )

    cdr_col = _pick_col(df_in, ["CDR", "Annual CDR", "Solved CDR"], required=False)
    target_cgl_col = _pick_col(df_in, ["Target CGL", "CGL", "Target CNL", "CNL"], required=False)
    severity_col = _pick_col(df_in, ["Severity", "Loss Severity"], required=False)
    dr_col = _pick_col(df_in, ["Discount Rate", "DR", "Disc Rate", "Yield"], required=False)
    cpr_col = _pick_col(df_in, ["CPR", "Annual CPR"], required=False)

    id_cols = []
    for aliases in (
        ["Repline ID", "Repline", "Pool ID", "Tier"],
        ["Loan ID", "Loan"],
        ["Purpose", "Interest Rate Band", "Loan purpose", "Fico bands", "LB / Non LB"],
        ["FICO", "Weighted Avg FICO"],
    ):
        c = _pick_col(df_in, aliases, required=False)
        if c is not None:
            id_cols.append(c)

    # Keep only rows that look like valid data rows.
    valid_mask = (
        df_in[upb_col].apply(_to_num).notna()
        & df_in[wac_col].apply(_to_num).notna()
        & df_in[term_col].apply(_to_num).notna()
    )
    df_work = df_in.loc[valid_mask].copy()

    out_rows = []

    for i, row in df_work.iterrows():
        upb = float(_to_num(row[upb_col]))
        wac = float(_as_rate(row[wac_col]))
        term = int(float(_to_num(row[term_col])))

        severity = float(_as_rate(row[severity_col])) if severity_col is not None and pd.notna(_to_num(row[severity_col])) else default_severity
        discount_rate = float(_as_rate(row[dr_col])) if dr_col is not None and pd.notna(_to_num(row[dr_col])) else default_discount_rate
        cpr = float(_as_rate(row[cpr_col])) if cpr_col is not None and pd.notna(_to_num(row[cpr_col])) else default_cpr

        solved_cdr = None
        input_cdr = None
        target_cgl = None

        if cdr_col is not None and pd.notna(_to_num(row[cdr_col])):
            input_cdr = float(_as_rate(row[cdr_col]))
            solved_cdr = input_cdr
        elif target_cgl_col is not None and pd.notna(_to_num(row[target_cgl_col])):
            target_cgl = float(_as_rate(row[target_cgl_col]))
            solved_cdr = goal_seek_cdr(upb, wac, term, severity, discount_rate, cpr, target_cgl)
        else:
            raise ValueError(
                f"Row {i}: provide either CDR or Target CGL."
            )

        _, summary = build_amort_table(
            upb=upb,
            wac=wac,
            term=term,
            severity=severity,
            discount_rate=discount_rate,
            cpr=cpr,
            cdr=solved_cdr,
        )

        yld = compute_yield(upb, wac, term, severity, cpr, solved_cdr, summary["Price"])

        out = {
            "Row": i,
            "UPB": upb,
            "WAC": wac,
            "Term": term,
            "Severity": severity,
            "Discount Rate": discount_rate,
            "CPR": cpr,
            "Input CDR": input_cdr,
            "Target CGL": target_cgl,
            "Solved CDR": solved_cdr,
            "Model Price": summary["Price"],
            "Model PV": summary["Total PV"],
            "Model CGL": summary["CGL"],
            "Model Yield": yld,
            "Total Interest": summary["Total Interest"],
            "Total Defaults": summary["Total Defaults"],
            "Total Recoveries": summary["Total Recoveries"],
            "Total Cash Flow": summary["Total Cash Flow"],
        }

        for c in id_cols:
            out[c] = row[c]

        out_rows.append(out)

    df_out = pd.DataFrame(out_rows)

    if output_path is None:
        output_path = file_path.with_name(file_path.stem + "_model_output.xlsx")
    output_path = Path(output_path)
    df_out.to_excel(output_path, index=False)

    print(f"Input file:  {file_path}")
    print(f"Rows run:    {len(df_out):,}")
    print(f"Output file: {output_path}")
    display(df_out.head(10))

    return df_out, output_path


# Use an explicit run cell to control which file is processed.

In [18]:
# Explicit run against the shared repline workbook
shared_file = r"G:\Shared drives\Capital Markets 2\Capital Markets - Dropbox\20. Projects\Loan pricing automation\replines fico lb purpose.xlsx"
results_df, results_path = run_replines_from_excel(file_path=shared_file)
print(results_path)

Input file:  G:\Shared drives\Capital Markets 2\Capital Markets - Dropbox\20. Projects\Loan pricing automation\replines fico lb purpose.xlsx
Rows run:    273
Output file: G:\Shared drives\Capital Markets 2\Capital Markets - Dropbox\20. Projects\Loan pricing automation\replines fico lb purpose_model_output.xlsx


,Row,UPB,WAC,Term,Severity,Discount Rate,CPR,Input CDR,Target CGL,Solved CDR,Model Price,Model PV,Model CGL,Model Yield,Total Interest,Total Defaults,Total Recoveries,Total Cash Flow,Tier,Loan purpose
0,0,"678,025.8900",0.0902,24,0.9000,0.0750,0.1200,None,0.0198,0.0201,0.9993,"677,533.5908",0.0198,0.0750,"59,582.4296","13,424.9126","1,342.4913","725,525.8982",1,NaN
1,1,"75,000.0000",0.0878,24,0.9000,0.0750,0.1200,None,0.0175,0.0178,0.9990,"74,927.3032",0.0175,0.0750,"6,419.9778","1,312.5000",131.2500,"80,238.7278",1,NaN
2,2,"95,000.0000",0.1068,24,0.9000,0.0750,0.1200,None,0.0120,0.0121,1.0217,"97,058.0660",0.0120,0.0750,"9,982.8373","1,140.0000",114.0000,"103,956.8373",1,NaN
3,3,"75,000.0000",0.0557,24,0.9000,0.0750,0.1200,None,0.0100,0.0102,0.9758,"73,183.9807",0.0100,0.0750,"4,055.0388",750.0000,75.0000,"78,380.0388",1,NaN
4,4,"65,000.0000",0.0612,24,0.9000,0.0750,0.1200,None,0.0100,0.0102,0.9809,"63,758.1432",0.0100,0.0750,"3,870.8943",650.0000,65.0000,"68,285.8943",1,NaN
5,5,"123,045.0000",0.0829,24,0.9000,0.0750,0.1200,None,0.0100,0.0102,1.0010,"123,172.2542",0.0100,0.0750,"9,988.5685","1,230.4500",123.0450,"131,926.1635",1,NaN
6,6,"199,820.5900",0.0724,36,0.9000,0.0750,0.1200,None,0.0241,0.0172,0.9794,"195,697.7500",0.0241,0.0750,"20,054.6751","4,815.6762",481.5676,"215,541.1565",1,NaN
7,7,"276,000.0000",0.0702,36,0.9000,0.0750,0.1200,None,0.0217,0.0155,0.9786,"270,088.6434",0.0217,0.0750,"26,888.4613","5,989.2000",598.9200,"297,498.1813",1,NaN
8,8,"40,000.0000",0.0851,36,0.9000,0.0750,0.1200,None,0.0157,0.0111,1.0031,"40,125.4054",0.0157,0.0750,"4,777.6522",628.0000,62.8000,"44,212.4522",1,NaN
9,9,"66,000.0000",0.0945,36,0.9000,0.0750,0.1200,None,0.0142,0.0100,1.0168,"67,108.6407",0.0142,0.0750,"8,796.6764",937.2000,93.7200,"73,953.1964",1,NaN


G:\Shared drives\Capital Markets 2\Capital Markets - Dropbox\20. Projects\Loan pricing automation\replines fico lb purpose_model_output.xlsx


In [7]:
# Inspect workbook structure for the shared file
shared_file = r"G:\Shared drives\Capital Markets 2\Capital Markets - Dropbox\20. Projects\Loan pricing automation\replines fico lb purpose.xlsx"
xl = pd.ExcelFile(shared_file)
print("Sheets:", xl.sheet_names)

for s in xl.sheet_names[:5]:
    print("\n--- Sheet:", s, "---")
    peek = pd.read_excel(shared_file, sheet_name=s, header=None, nrows=12)
    display(peek)

Sheets: ['Pricing Tier-Term-Fico-Channel']

--- Sheet: Pricing Tier-Term-Fico-Channel ---


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
0,NaN,Pricing factors,NaN,NaN,NaN,1,2,3,4,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"25,038,837.0000"
1,1.0000,Tier,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2.0000,Term,NaN,RA Loss,DV01,Tier,Term,Fico bands,LB / Non LB,Loan purpose,WA WAC,CGL,Price (Tier + Term + fico ),Total Current Principal,UPB%,NaN,NaN,NaN,Weighted Avg CGL,Weighted Avg CGL,NaN
3,3.0000,Fico band,0.0100,0.0100,0.0085,1,24,700-719,LB+Non LB,NaN,9.0222,0.0198,1.0107,"678,025.8900",0.0271,NaN,NaN,NaN,0.0100,NaN,NaN
4,4.0000,LB/Non LB,NaN,NaN,NaN,1,24,720-739,LB+Non LB,NaN,8.7800,0.0175,1.0107,75000,0.0030,NaN,NaN,NaN,NaN,NaN,NaN
5,5.0000,Loan purpose,NaN,0.0100,0.0085,1,24,740-759,LB+Non LB,NaN,10.6755,0.0120,1.0315,95000,0.0038,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,1,24,760-779,LB+Non LB,NaN,5.5683,0.0100,0.9910,75000,0.0030,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,0.0100,0.0085,1,24,780-800,LB+Non LB,NaN,6.1231,0.0100,0.9955,65000,0.0026,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,0.0100,0.0085,1,24,>800,LB+Non LB,NaN,8.2930,0.0100,1.0134,123045,0.0049,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,0.0200,0.0100,0.0085,1,36,720-739,LB+Non LB,NaN,7.2352,0.0241,0.9968,"199,820.5900",0.0080,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Compact schema probe
shared_file = r"G:\Shared drives\Capital Markets 2\Capital Markets - Dropbox\20. Projects\Loan pricing automation\replines fico lb purpose.xlsx"
raw = pd.read_excel(shared_file, sheet_name=0, header=None, nrows=20)
print("Shape (20 rows sample):", raw.shape)
for i in range(min(12, len(raw))):
    row_vals = [str(v) for v in raw.iloc[i].tolist()]
    row_vals = [v for v in row_vals if v != "nan"]
    print(i, row_vals[:12])

Shape (20 rows sample): (20, 21)
0 ['Pricing factors', '1', '2', '3', '4', '5', '25038837.0']
1 ['1.0', 'Tier']
2 ['2.0', 'Term', 'RA Loss ', 'DV01', 'Tier', 'Term', 'Fico bands', 'LB / Non LB', 'Loan purpose', 'WA WAC', 'CGL', 'Price (Tier + Term + fico )']
3 ['3.0', 'Fico band', '0.01', '0.01', '0.0085', '1', '24', '700-719', 'LB+Non LB', '9.022246', '0.0198', '1.010685']
4 ['4.0', 'LB/Non LB', '1', '24', '720-739', 'LB+Non LB', '8.78', '0.0175', '1.010743', '75000', '0.0029953467886707358']
5 ['5.0', 'Loan purpose', '0.01', '0.0085', '1', '24', '740-759', 'LB+Non LB', '10.67553', '0.012', '1.031487', '95000']
6 ['1', '24', '760-779', 'LB+Non LB', '5.568333', '0.01', '0.990964', '75000', '0.0029953467886707358']
7 ['0.01', '0.0085', '1', '24', '780-800', 'LB+Non LB', '6.123077', '0.01', '0.995507', '65000', '0.002595967216847971']
8 ['0.01', '0.0085', '1', '24', '>800', 'LB+Non LB', '8.293009', '0.01', '1.013408', '123045', '0.004914165941493209']
9 ['0.02', '0.01', '0.0085', '1', '3

In [9]:
# Try reading with header row offset used by this workbook
shared_file = r"G:\Shared drives\Capital Markets 2\Capital Markets - Dropbox\20. Projects\Loan pricing automation\replines fico lb purpose.xlsx"
df_h2 = pd.read_excel(shared_file, sheet_name=0, header=2)
print("Columns:", list(df_h2.columns))
print("Rows:", len(df_h2))
display(df_h2.head(10))

Columns: [2, 'Term', 'Unnamed: 2', 'RA Loss ', 'DV01', 'Tier', 'Term.1', 'Fico bands', 'LB / Non LB', 'Loan purpose', 'WA WAC', 'CGL', 'Price (Tier + Term + fico )', 'Total Current Principal', 'UPB%', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Weighted Avg CGL', 'Weighted Avg CGL.1', 'Unnamed: 20']
Rows: 273


,2,Term,Unnamed: 2,RA Loss,DV01,Tier,Term.1,Fico bands,LB / Non LB,Loan purpose,WA WAC,CGL,Price (Tier + Term + fico ),Total Current Principal,UPB%,Unnamed: 15,Unnamed: 16,Unnamed: 17,Weighted Avg CGL,Weighted Avg CGL.1,Unnamed: 20
0,3.0000,Fico band,0.0100,0.0100,0.0085,1,24,700-719,LB+Non LB,NaN,9.0222,0.0198,1.0107,"678,025.8900",0.0271,NaN,NaN,NaN,0.0100,NaN,NaN
1,4.0000,LB/Non LB,NaN,NaN,NaN,1,24,720-739,LB+Non LB,NaN,8.7800,0.0175,1.0107,"75,000.0000",0.0030,NaN,NaN,NaN,NaN,NaN,NaN
2,5.0000,Loan purpose,NaN,0.0100,0.0085,1,24,740-759,LB+Non LB,NaN,10.6755,0.0120,1.0315,"95,000.0000",0.0038,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,1,24,760-779,LB+Non LB,NaN,5.5683,0.0100,0.9910,"75,000.0000",0.0030,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,0.0100,0.0085,1,24,780-800,LB+Non LB,NaN,6.1231,0.0100,0.9955,"65,000.0000",0.0026,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,0.0100,0.0085,1,24,>800,LB+Non LB,NaN,8.2930,0.0100,1.0134,"123,045.0000",0.0049,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,0.0200,0.0100,0.0085,1,36,720-739,LB+Non LB,NaN,7.2352,0.0241,0.9968,"199,820.5900",0.0080,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,0.0100,0.0085,1,36,740-759,LB+Non LB,NaN,7.0174,0.0217,0.9966,"276,000.0000",0.0110,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,0.0100,0.0085,1,36,760-779,LB+Non LB,NaN,8.5100,0.0157,1.0182,"40,000.0000",0.0016,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,0.0100,0.0085,1,36,780-800,LB+Non LB,NaN,9.4474,0.0142,1.0299,"66,000.0000",0.0026,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# Diagnose key columns after header detection on shared file
shared_file = r"G:\Shared drives\Capital Markets 2\Capital Markets - Dropbox\20. Projects\Loan pricing automation\replines fico lb purpose.xlsx"
probe = _read_excel_detect_header(shared_file, 0)
print("Detected columns:")
print(list(probe.columns))

c_upb = _pick_col(probe, ["UPB", "Current UPB", "Total Current Principal", "Total Orig Principal"], required=False)
c_wac = _pick_col(probe, ["WAC", "Weighted Avg Interest", "WA WAC"], required=False)
c_term = _pick_col(probe, ["Term", "Term.1", "Weighted Avg Term"], required=False)
c_cgl = _pick_col(probe, ["CGL", "Target CGL"], required=False)

print("Selected:", c_upb, c_wac, c_term, c_cgl)
if c_upb is not None:
    print("UPB numeric count:", probe[c_upb].apply(_to_num).notna().sum())
if c_wac is not None:
    print("WAC numeric count:", probe[c_wac].apply(_to_num).notna().sum())
if c_term is not None:
    print("Term numeric count:", probe[c_term].apply(_to_num).notna().sum())
if c_cgl is not None:
    print("CGL numeric count:", probe[c_cgl].apply(_to_num).notna().sum())

cols_to_show = [c for c in [c_term, c_wac, c_cgl, c_upb] if c is not None]
print("Sample values:")
display(probe[cols_to_show].head(30))

Detected columns:
[2, 'Term', 'Unnamed: 2', 'RA Loss ', 'DV01', 'Tier', 'Term.1', 'Fico bands', 'LB / Non LB', 'Loan purpose', 'WA WAC', 'CGL', 'Price (Tier + Term + fico )', 'Total Current Principal', 'UPB%', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Weighted Avg CGL', 'Weighted Avg CGL.1', 'Unnamed: 20']
Selected: UPB% WA WAC Term CGL
UPB numeric count: 273
WAC numeric count: 273
Term numeric count: 0
CGL numeric count: 273
Sample values:


,Term,WA WAC,CGL,UPB%
0,Fico band,9.0222,0.0198,0.0271
1,LB/Non LB,8.7800,0.0175,0.0030
2,Loan purpose,10.6755,0.0120,0.0038
3,NaN,5.5683,0.0100,0.0030
4,NaN,6.1231,0.0100,0.0026
5,NaN,8.2930,0.0100,0.0049
6,NaN,7.2352,0.0241,0.0080
7,NaN,7.0174,0.0217,0.0110
8,NaN,8.5100,0.0157,0.0016
9,NaN,9.4474,0.0142,0.0026


In [24]:
from openpyxl import load_workbook
from pathlib import Path
import shutil
from datetime import datetime


def overwrite_price_column_in_workbook(
    file_path,
    sheet_name="Pricing Tier-Term-Fico-Channel",
    backup=True,
    default_severity=0.90,
    default_discount_rate=0.075,
    default_cpr=0.12,
    force_discount_rate=None,
):
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    if backup:
        backup_path = file_path.with_name(file_path.stem + "_backup_before_price_overwrite" + file_path.suffix)
        shutil.copy2(file_path, backup_path)
        print(f"Backup created: {backup_path}")

    wb = load_workbook(file_path)
    ws = wb[sheet_name] if sheet_name in wb.sheetnames else wb[wb.sheetnames[0]]

    def norm(x):
        return _norm_col(x)

    def find_header_row(max_scan_rows=20):
        upb_aliases = {
            norm("Total Current Principal"),
            norm("Current Principal"),
            norm("Total Orig Principal"),
            norm("Current UPB"),
            norm("UPB"),
        }
        wac_aliases = {
            norm("WA WAC"),
            norm("Weighted Avg Interest"),
            norm("WAC"),
            norm("Interest Rate"),
        }
        term_aliases = {
            norm("Term"),
            norm("Term.1"),
            norm("Weighted Avg Term"),
            norm("Remaining Term"),
        }

        for r in range(1, min(max_scan_rows, ws.max_row) + 1):
            row_vals = {norm(ws.cell(r, c).value) for c in range(1, ws.max_column + 1)}
            if any(a in row_vals for a in upb_aliases) and any(a in row_vals for a in wac_aliases) and any(a in row_vals for a in term_aliases):
                return r
        raise ValueError("Could not detect header row in worksheet.")

    header_row = find_header_row()

    header_map = {}
    for c in range(1, ws.max_column + 1):
        header_map[norm(ws.cell(header_row, c).value)] = c

    def pick_col(aliases, required=True):
        for a in aliases:
            k = norm(a)
            if k in header_map:
                return header_map[k]
        if required:
            raise KeyError(f"Missing required column. Tried aliases: {aliases}")
        return None

    upb_col = pick_col([
        "Total Current Principal",
        "Current Principal",
        "Total Orig Principal",
        "Current UPB",
        "Original Balance",
        "Balance",
        "UPB",
    ])
    wac_col = pick_col([
        "WA WAC",
        "Weighted Avg Interest",
        "WA Interest",
        "WAC",
        "Coupon",
        "Rate",
        "Interest Rate",
    ])
    term_col = pick_col([
        "Term.1",
        "Weighted Avg Term",
        "Remaining Term",
        "Amort Term",
        "Amortization Term",
        "Term",
    ])
    cdr_col = pick_col(["CDR", "Annual CDR", "Solved CDR"], required=False)
    cgl_col = pick_col(["Target CGL", "CGL", "Target CNL", "CNL"], required=False)
    severity_col = pick_col(["Severity", "Loss Severity"], required=False)
    dr_col = pick_col(["Discount Rate", "DR", "Disc Rate", "Yield"], required=False)
    cpr_col = pick_col(["CPR", "Annual CPR"], required=False)

    price_col = pick_col([
        "Price ( tier + term  + fico",
        "Price ( tier + term + fico",
        "Price (Tier + Term + fico )",
        "Price",
    ], required=False)

    if price_col is None:
        price_col = ws.max_column + 1
        ws.cell(header_row, price_col).value = "Price ( tier + term  + fico"

    solved_cdr_col = pick_col(["Solved CDR", "Goal Seeked CDR", "Goal-Sought CDR"], required=False)
    if solved_cdr_col is None:
        solved_cdr_col = ws.max_column + 1
        ws.cell(header_row, solved_cdr_col).value = "Solved CDR"

    rows_updated = 0
    for r in range(header_row + 1, ws.max_row + 1):
        upb = _to_num(ws.cell(r, upb_col).value)
        wac = _as_rate(ws.cell(r, wac_col).value)
        term = _to_num(ws.cell(r, term_col).value)

        if pd.isna(upb) or pd.isna(wac) or pd.isna(term):
            continue

        severity = _as_rate(ws.cell(r, severity_col).value) if severity_col is not None else default_severity
        cpr = _as_rate(ws.cell(r, cpr_col).value) if cpr_col is not None else default_cpr

        if force_discount_rate is not None:
            discount_rate = float(force_discount_rate)
        else:
            discount_rate = _as_rate(ws.cell(r, dr_col).value) if dr_col is not None else default_discount_rate

        if pd.isna(severity):
            severity = default_severity
        if pd.isna(discount_rate):
            discount_rate = default_discount_rate
        if pd.isna(cpr):
            cpr = default_cpr

        cdr_val = _as_rate(ws.cell(r, cdr_col).value) if cdr_col is not None else np.nan
        cgl_val = _as_rate(ws.cell(r, cgl_col).value) if cgl_col is not None else np.nan

        if not pd.isna(cdr_val):
            solved_cdr = float(cdr_val)
        elif not pd.isna(cgl_val):
            solved_cdr = float(
                goal_seek_cdr(
                    float(upb),
                    float(wac),
                    int(float(term)),
                    float(severity),
                    float(discount_rate),
                    float(cpr),
                    float(cgl_val),
                )
            )
        else:
            continue

        _, summary = build_amort_table(
            upb=float(upb),
            wac=float(wac),
            term=int(float(term)),
            severity=float(severity),
            discount_rate=float(discount_rate),
            cpr=float(cpr),
            cdr=solved_cdr,
        )

        ws.cell(r, price_col).value = float(summary["Price"])
        ws.cell(r, price_col).number_format = "0.000000"
        ws.cell(r, solved_cdr_col).value = float(solved_cdr)
        ws.cell(r, solved_cdr_col).number_format = "0.0000000000"
        rows_updated += 1

    saved_path = None
    save_candidates = [file_path]

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_candidates.append(file_path.with_name(file_path.stem + f"_UPDATED_PRICES_{ts}" + file_path.suffix))
    save_candidates.append(Path.home() / "Downloads" / (file_path.stem + f"_UPDATED_PRICES_{ts}" + file_path.suffix))

    for candidate in save_candidates:
        try:
            wb.save(candidate)
            saved_path = candidate
            break
        except PermissionError:
            continue

    if saved_path is None:
        raise PermissionError("Could not save workbook to shared drive or local Downloads; all candidate paths are locked.")

    print(f"Saved file: {saved_path}")
    print(f"Sheet used: {ws.title}")
    print(f"Header row: {header_row}")
    print(f"Rows updated in price column: {rows_updated}")
    print(f"Discount rate used: {force_discount_rate if force_discount_rate is not None else 'row/default'}")
    print(f"Solved CDR column: {ws.cell(header_row, solved_cdr_col).value}")
    return saved_path, rows_updated


shared_file = r"G:\Shared drives\Capital Markets 2\Capital Markets - Dropbox\20. Projects\Loan pricing automation\replines fico lb purpose.xlsx"
saved_path, rows_updated = overwrite_price_column_in_workbook(
    shared_file,
    force_discount_rate=0.0575,
)
print(saved_path, rows_updated)

Backup created: G:\Shared drives\Capital Markets 2\Capital Markets - Dropbox\20. Projects\Loan pricing automation\replines fico lb purpose_backup_before_price_overwrite.xlsx
Saved file: G:\Shared drives\Capital Markets 2\Capital Markets - Dropbox\20. Projects\Loan pricing automation\replines fico lb purpose_UPDATED_PRICES_20260415_111950.xlsx
Sheet used: Pricing Tier-Term-Fico-Channel
Header row: 3
Rows updated in price column: 273
Discount rate used: 0.0575
Solved CDR column: Solved CDR
G:\Shared drives\Capital Markets 2\Capital Markets - Dropbox\20. Projects\Loan pricing automation\replines fico lb purpose_UPDATED_PRICES_20260415_111950.xlsx 273
